# 02 - Feature Engineering
## AEGIS: Enterprise AI Governance & Fraud Intelligence Platform

---

## Notebook Goal

Transform raw transaction data into a feature set optimized for XGBoost fraud detection. Features are engineered based on insights from notebook 01 EDA.

---

## Features to Engineer

| Feature | Type | Source | Insight from EDA |
|---------|------|--------|-----------------|
| `geo_distance` | Numeric (km) | lat, long, merch_lat, merch_long | Transaction far from home = fraud signal |
| `age` | Numeric | dob, trans_date | Age at time of transaction |
| `transactions_last_1h` | Numeric | cc_num, datetime | Velocity = strong fraud signal |
| `transactions_last_24h` | Numeric | cc_num, datetime | Daily velocity pattern |
| `amt_last_1h` | Numeric | cc_num, amt, datetime | Amount velocity per card |
| `unique_merchants_1h` | Numeric | cc_num, merchant, datetime | Multiple merchants = card testing |
| `log_amt` | Numeric | amt | Handle extreme skew |
| `is_night` | Binary | trans_hour | Fraud peaks 22:00 to 23:00 |
| `is_friday` | Binary | trans_dayofweek | Friday highest fraud day |
| `is_weekend` | Binary | trans_dayofweek | Weekend gap exploitation |

---

## Features to Drop

| Feature | Reason |
|---------|--------|
| `state`, `city`, `zip`, `street`, `city_pop` | Weak signal or location detail replaced by `geo_distance` |
| `job` | No predictive value |
| `gender` | Weak signal, potential bias |
| `lat`, `long`, `merch_lat`, `merch_long` | Replaced by `geo_distance` |
| `cc_num` | Used for velocity then dropped |
| `merchant` | Used for `unique_merchants_1h` then dropped |
| `first`, `last`, `trans_num` | Identifiers, not predictive |
| `dob`, `trans_date_trans_time`, `unix_time` | Raw timestamps, replaced by derived features |

---

## Encoding Plan

| Feature | Encoding |
|---------|----------|
| `category` | One hot encoding, 14 categories |
| `trans_dayofweek` | One hot encoding, 7 days |
| `trans_hour` | Keep numeric |
| `trans_month` | Keep numeric |

---

## Leakage Note

Velocity features look only at transactions **strictly before** the current one (`closed='left'` in the rolling windows). Using the current or future transactions would leak the label into the features.

**Assumed input:** the Kaggle synthetic dataset `fraudTrain.csv` placed at `data/fraudTrain.csv`. Adjust `DATA_PATH` if yours is elsewhere.

In [12]:
# Run once if pyproj is not installed
# pip install pyproj

In [13]:
import numpy as np
import pandas as pd
from pyproj import Geod
import os
from pathlib import Path

pd.set_option('display.max_columns', 60)

## 1. Load Raw Data

In [14]:
DATA_PATH = r"..\data\raw\fraudTrain.csv"
df = pd.read_csv(DATA_PATH)
df['trans_dt'] = pd.to_datetime(df['trans_date_trans_time'])
df['dob'] = pd.to_datetime(df['dob'])

print(df.shape)
df.head(3)

(1296675, 24)


,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,state,zip,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud,trans_dt
0,0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,Moravian Falls,NC,28654,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0,2019-01-01 00:00:18
1,1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,Orient,WA,99160,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0,2019-01-01 00:00:44
2,2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,Malad City,ID,83252,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0,2019-01-01 00:00:51


## 2. Velocity Features (Per Card, Past Transactions Only)

Sort by card and time first. Rolling windows use `closed='left'` so the current transaction is excluded.

In [15]:
df = df.sort_values(['cc_num', 'trans_dt']).reset_index(drop=True)


def rolling_per_card(col, window, func):
    roll = (
        df.set_index('trans_dt')
          .groupby('cc_num')[col]
          .rolling(window, closed='left')
    )
    return getattr(roll, func)().to_numpy()


df['transactions_last_1h'] = rolling_per_card('amt', '1h', 'count')
df['transactions_last_24h'] = rolling_per_card('amt', '24h', 'count')
df['amt_last_1h'] = np.nan_to_num(rolling_per_card('amt', '1h', 'sum'))

df[['transactions_last_1h', 'transactions_last_24h', 'amt_last_1h']].describe()

,transactions_last_1h,transactions_last_24h,amt_last_1h
count,212547.000000,1.187756e+06,1.296675e+06
mean,1.156281,4.240331e+00,1.455956e+01
std,0.427784,2.976590e+00,9.658414e+01
min,1.000000,1.000000e+00,0.000000e+00
25%,1.000000,2.000000e+00,0.000000e+00
50%,1.000000,4.000000e+00,0.000000e+00
75%,1.000000,6.000000e+00,0.000000e+00
max,6.000000,3.500000e+01,2.739012e+04


### Unique merchants in the last hour

Pandas rolling windows do not support `nunique`, so this uses a two-pointer sliding window per card. It runs in plain Python and may take 10 to 30 seconds on the full dataset.

In [16]:
def count_unique_merchants(times, merchants, window_seconds=3600):
    """For each transaction, count distinct merchants in the previous window_seconds."""
    out = np.zeros(len(times), dtype=np.int32)
    seen = {}
    left = 0
    for right in range(len(times)):
        while times[left] < times[right] - window_seconds:
            m = merchants[left]
            seen[m] -= 1
            if seen[m] == 0:
                del seen[m]
            left += 1
        out[right] = len(seen)
        m = merchants[right]
        seen[m] = seen.get(m, 0) + 1
    return out


unix_times = df['unix_time'].to_numpy()
merchant_vals = df['merchant'].to_numpy()
unique_merch = np.zeros(len(df), dtype=np.int32)

for cc, idx in df.groupby('cc_num').indices.items():
    unique_merch[idx] = count_unique_merchants(unix_times[idx], merchant_vals[idx])

df['unique_merchants_1h'] = unique_merch
df['unique_merchants_1h'].describe()

count    1.296675e+06
mean     1.900283e-01
std      4.624653e-01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      6.000000e+00
Name: unique_merchants_1h, dtype: float64

## 3. Geo Distance (Cardholder to Merchant)

`pyproj.Geod` computes the geodesic distance on the WGS84 ellipsoid. Output is converted from metres to kilometres.

In [17]:
geod = Geod(ellps='WGS84')

_, _, dist_m = geod.inv(
    df['long'].to_numpy(), df['lat'].to_numpy(),
    df['merch_long'].to_numpy(), df['merch_lat'].to_numpy()
)
df['geo_distance'] = dist_m / 1000.0

df['geo_distance'].describe()

count    1.296675e+06
mean     7.611248e+01
std      2.909270e+01
min      2.227351e-02
25%      5.535784e+01
50%      7.826335e+01
75%      9.846835e+01
max      1.518682e+02
Name: geo_distance, dtype: float64

## 4. Age, Amount, and Time Features

In [18]:
# Age at time of transaction
df['age'] = ((df['trans_dt'] - df['dob']).dt.days / 365.25).astype(int)

# Log transform for the heavily skewed amount
df['log_amt'] = np.log1p(df['amt'])

# Time components
df['trans_hour'] = df['trans_dt'].dt.hour
df['trans_dayofweek'] = df['trans_dt'].dt.dayofweek   # 0 = Monday, 6 = Sunday
df['trans_month'] = df['trans_dt'].dt.month

# Binary flags
df['is_night'] = df['trans_hour'].isin([22, 23, 0, 1, 2, 3, 4, 5]).astype(int)
df['is_friday'] = (df['trans_dayofweek'] == 4).astype(int)
df['is_weekend'] = (df['trans_dayofweek'] >= 5).astype(int)

df[['age', 'log_amt', 'is_night', 'is_friday', 'is_weekend']].describe()

,age,log_amt,is_night,is_friday,is_weekend
count,1.296675e+06,1.296675e+06,1.296675e+06,1.296675e+06,1.296675e+06
mean,4.549541e+01,3.533470e+00,2.999333e-01,1.174327e-01,3.482260e-01
std,1.739698e+01,1.289448e+00,4.582286e-01,3.219353e-01,4.764083e-01
min,1.300000e+01,6.931472e-01,0.000000e+00,0.000000e+00,0.000000e+00
25%,3.200000e+01,2.365560e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,4.300000e+01,3.881976e+00,0.000000e+00,0.000000e+00,0.000000e+00
75%,5.700000e+01,4.432482e+00,1.000000e+00,0.000000e+00,1.000000e+00
max,9.500000e+01,1.027332e+01,1.000000e+00,1.000000e+00,1.000000e+00


## 5. Drop Columns and Encode Categoricals

In [19]:
drop_cols = [
    'state', 'city', 'zip', 'street', 'city_pop', 'job', 'gender',
    'lat', 'long', 'merch_lat', 'merch_long',
    'cc_num', 'merchant', 'first', 'last', 'trans_num',
    'dob', 'trans_date_trans_time', 'unix_time', 'trans_dt'
]

features = df.drop(columns=drop_cols)

features = pd.get_dummies(features, columns=['category'], prefix='cat', dtype=int)
features = pd.get_dummies(features, columns=['trans_dayofweek'], prefix='dow', dtype=int)

# Move the target to the last column
target = features.pop('is_fraud')
features['is_fraud'] = target

print('Final shape:', features.shape)
print('Columns:', list(features.columns))

Final shape: (1296675, 36)
Columns: ['Unnamed: 0', 'amt', 'transactions_last_1h', 'transactions_last_24h', 'amt_last_1h', 'unique_merchants_1h', 'geo_distance', 'age', 'log_amt', 'trans_hour', 'trans_month', 'is_night', 'is_friday', 'is_weekend', 'cat_entertainment', 'cat_food_dining', 'cat_gas_transport', 'cat_grocery_net', 'cat_grocery_pos', 'cat_health_fitness', 'cat_home', 'cat_kids_pets', 'cat_misc_net', 'cat_misc_pos', 'cat_personal_care', 'cat_shopping_net', 'cat_shopping_pos', 'cat_travel', 'dow_0', 'dow_1', 'dow_2', 'dow_3', 'dow_4', 'dow_5', 'dow_6', 'is_fraud']


## 6. Sanity Checks

In [20]:
print('Missing values:', features.isna().sum().sum())
print('Fraud rate overall:', round(features['is_fraud'].mean() * 100, 3), '%')

# Confirm the EDA signal: fraud rate by night flag
features.groupby('is_night')['is_fraud'].mean().mul(100).round(3).rename('fraud_rate_%')

Missing values: 1193047
Fraud rate overall: 0.579 %


is_night
0    0.114
1    1.663
Name: fraud_rate_%, dtype: float64

In [21]:
features.describe().T

,count,mean,std,min,25%,50%,75%,max
Unnamed: 0,1296675.0,648337.000000,374317.974488,0.000000,324168.500000,648337.000000,972505.500000,1.296674e+06
amt,1296675.0,70.351035,160.316039,1.000000,9.650000,47.520000,83.140000,2.894890e+04
transactions_last_1h,212547.0,1.156281,0.427784,1.000000,1.000000,1.000000,1.000000,6.000000e+00
transactions_last_24h,1187756.0,4.240331,2.976590,1.000000,2.000000,4.000000,6.000000,3.500000e+01
amt_last_1h,1296675.0,14.559564,96.584141,0.000000,0.000000,0.000000,0.000000,2.739012e+04
unique_merchants_1h,1296675.0,0.190028,0.462465,0.000000,0.000000,0.000000,0.000000,6.000000e+00
geo_distance,1296675.0,76.112479,29.092700,0.022274,55.357839,78.263353,98.468348,1.518682e+02
age,1296675.0,45.495406,17.396980,13.000000,32.000000,43.000000,57.000000,9.500000e+01
log_amt,1296675.0,3.533470,1.289448,0.693147,2.365560,3.881976,4.432482,1.027332e+01
trans_hour,1296675.0,12.804858,6.817824,0.000000,7.000000,14.000000,19.000000,2.300000e+01


## 7. Save Feature Set

In [23]:
OUTPUT_PATH = r'..\data\processed\fraudTrain_features.csv'

features.to_csv(OUTPUT_PATH, index=False)
print('Saved', features.shape[0], 'rows to', OUTPUT_PATH)

Saved 1296675 rows to ..\data\processed\fraudTrain_features.csv
